# Week 7 Notebook: FraudFinder Bank — Training and Interpreting a Neural Network

*OMAIB · Module 1 · Week 7*

*© H. Sami Karaca. Questrom School of Business, Boston University.
Online Master of Science in Artificial Intelligence in Business.
Licensed for use in this course only; not for redistribution.*

You are a manager at FraudFinder Bank. Your team proposes a neural network that predicts a transaction's **fraud suspicion score, from 0 to 100**. The dataset is a sample of **1,500 transactions** drawn from FraudFinder's millions of historical rows. The data used in this activity is **synthetic** — built and verified for the course, so no real customer information is involved.

Watch a neural network learn, compare seven designs, and decide what evidence you would
take to the board. **The optional activity in the notebook can be skipped, and you may complete either interpretation path or both.**

**Read the errors:** RMSE measures prediction error in score points; lower is better. Training RMSE uses the examples the model learned from. Validation RMSE uses held-out examples. The **gap = validation RMSE − training RMSE**.

**Read the score:** a prediction **above 75** triggers decline and review; 75 or below is allowed. This synthetic suspicion index is not a probability or a confirmed fraud outcome. This is how most fraud systems work: score first, then apply a threshold.

## Start in Colab

Before running any cells, select **File › Save a copy in Drive** so your work saves to your own Google Drive. This notebook loads everything it needs automatically — there's nothing for you to upload. Run the cells in order. This version uses the libraries already in your session. Results may vary with the runtime; use your own output in your answers.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from IPython.display import display, Markdown, HTML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning
from threadpoolctl import threadpool_limits

warnings.filterwarnings("ignore", category=DeprecationWarning)

## Meet the data

The sample contains **1,500 synthetic transactions**, 14 predictors, and a
reference score. All predictors are available before the current transaction.
Historical counts and recency describe earlier authorization attempts on
the same card account, including declined attempts.

In [ ]:
# Fetch the dataset from the course's public repository
DATA_URL = ('https://raw.githubusercontent.com/hskaraca/BU-OMAIB-Public/'
            'main/Module1/Week7/Mod1_Wk07_FraudFinder_Data.csv')

df = pd.read_csv(DATA_URL)

display(df.head())

| Variable | Meaning |
|---|---|
| `amount` | Purchase amount in US dollars |
| `hour_of_day` | Hour, 0–23, in the cardholder's home timezone |
| `day_of_week` | Day in that timezone: Monday=0 through Sunday=6 |
| `days_since_last_txn` | Days since the previous attempt; 0.25 means six hours |
| `txn_count_24h` | Prior attempts in the last 24 hours, excluding the current one |
| `txn_count_7d` | Prior attempts in seven days, including the 24-hour count |
| `distance_from_home_km` | Home-to-terminal distance; for online purchases, approximate device/IP-origin distance |
| `account_tenure_months` | Completed months since account opening; zero means less than a month |
| `cardholder_age` | Age in completed years |
| `foreign_transaction` | 1 if the merchant/acquirer is in another country; otherwise 0 |
| `merchant_category` | Merchant business: Electronics, Gas, General Retail, Grocery, Jewelry, Restaurant, Travel, or Other |
| `device_type` | Payment channel: Chip, Swipe, Contactless, or Online |
| `income_tier` | Income category: Low, Med, or High |
| `online_transaction` | 1 for the Online channel; otherwise 0 |
| `risk_score` | Target: synthetic reference suspicion score, 0–100 |

In [ ]:
print(f'{len(df):,} transactions; score range {df.risk_score.min():.1f}–{df.risk_score.max():.1f}')
print(f'Reference scores above 75: {(df.risk_score > 75).sum()} ({(df.risk_score > 75).mean():.1%})')
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(df.risk_score, bins=np.linspace(0, 100, 41), color='steelblue', edgecolor='white')
ax.axvline(75, color='firebrick', linestyle='--', label='Action threshold: >75')
ax.set(xlabel='Reference suspicion score', ylabel='Transactions', xlim=(0, 100))
ax.legend()
plt.tight_layout()
plt.show()

**Check your reading.** Where do most scores fall? Why might a small prediction
error matter more for a transaction near 75 than for one far below it?

## Prepare the data

Split the rows **70/30** into training and validation. Encode categories as
0/1 indicators, and scale using **training data only**. The Online channel
already has its own indicator, so we omit its duplicate dummy column.

In [ ]:
train_idx, valid_idx = train_test_split(np.arange(len(df)), test_size=0.3, random_state=1)
category_levels = {
    'merchant_category': ['Electronics', 'Gas', 'General Retail', 'Grocery', 'Jewelry', 'Other', 'Restaurant', 'Travel'],
    'device_type': ['Chip', 'Contactless', 'Online', 'Swipe'],
    'income_tier': ['High', 'Low', 'Med'],
}
X = df.drop(columns='risk_score').copy()
for name, levels in category_levels.items():
    X[name] = pd.Categorical(X[name], categories=levels)
X = pd.get_dummies(X, columns=list(category_levels), drop_first=True, dtype=float)
X = X.drop(columns='device_type_Online')
y = df.risk_score.to_numpy(dtype=float)
scaler_X, scaler_y = MinMaxScaler(), MinMaxScaler()
X_train = scaler_X.fit_transform(X.iloc[train_idx])
X_valid = scaler_X.transform(X.iloc[valid_idx])
y_train = scaler_y.fit_transform(y[train_idx].reshape(-1, 1)).ravel()
print(f'Training: {len(train_idx):,}; validation: {len(valid_idx):,}; encoded inputs: {X.shape[1]}')

# Movement 1 — Watch a Small Network Learn

Start with **one hidden layer and two sigmoid neurons**. Each run starts
from the same initialization. Give it 1, 5, 30, and 100 optimizer iterations
and compare the results. A final run allows up to 1,000 iterations for the
full-budget baseline; the optimizer may stop earlier.

The model adjusts weights to reduce training error. We convert its predictions
back into score points before calculating RMSE.

In [ ]:
# Common training settings; the architecture and iteration budget vary below.
settings = dict(solver='lbfgs', random_state=1, max_fun=15000, tol=1e-4, alpha=1e-4)
learning_runs = []
for budget in [1, 5, 30, 100, 1000]:
    model = MLPRegressor(hidden_layer_sizes=(2,), activation='logistic', max_iter=budget, **settings)
    with warnings.catch_warnings(record=True) as notices, threadpool_limits(limits=1):
        model.fit(X_train, y_train)
        pt = scaler_y.inverse_transform(model.predict(X_train).reshape(-1, 1)).ravel()
        pv = scaler_y.inverse_transform(model.predict(X_valid).reshape(-1, 1)).ravel()
    learning_runs.append({'budget': budget, 'model': model, 'pred_valid': pv,
                         'train_rmse': np.sqrt(mean_squared_error(y[train_idx], pt)),
                         'valid_rmse': np.sqrt(mean_squared_error(y[valid_idx], pv)), 'notices': list(notices)})
    for notice in notices:
        if not (issubclass(notice.category, ConvergenceWarning) and 'LIMIT' in str(notice.message).upper()):
            print(f'Budget {budget}: {notice.category.__name__}: {notice.message}')
snapshot_table = pd.DataFrame(learning_runs[:4])[['budget', 'train_rmse', 'valid_rmse']]
snapshot_table.columns = ['Iteration limit', 'Training RMSE', 'Validation RMSE']
display(snapshot_table.round(2))
if any(issubclass(w.category, ConvergenceWarning) and 'LIMIT' in str(w.message).upper() for r in learning_runs[:4] for w in r['notices']):
    print('Iteration-limit warnings are expected in these deliberately short runs.')

**Check your reading.** Compare the first and last rows. How much did training
and validation error change? Did all of the improvement happen at the same
rate? Validation error need not decrease at every step, even when training
error improves. A run that reaches its iteration limit is not automatically
a converged solution.

In [ ]:
all_scores = np.concatenate([y[valid_idx], *[r['pred_valid'] for r in learning_runs[:4]], [0., 100.]])
pad = max(2., (all_scores.max() - all_scores.min()) * 0.04)
limits = (all_scores.min() - pad, all_scores.max() + pad)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, run in zip(axes, learning_runs[:4]):
    ax.scatter(y[valid_idx], run['pred_valid'], alpha=0.4, color='steelblue', s=20)
    ax.plot(limits, limits, 'r--', linewidth=1.3)
    ax.set(title=f"Limit {run['budget']}\nValidation RMSE: {run['valid_rmse']:.2f}",
           xlabel='Reference score', ylabel='Predicted score', xlim=limits, ylim=limits)
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

**Check your reading.** Compare the first and last plots. Predictions closer
to the diagonal are more accurate. Where did the largest improvement occur?
More training does not have to improve validation error at every step.

Now read the simple model's result with the full training budget.

In [ ]:
simple_record = learning_runs[-1]
print(f"Simple sigmoid model — training RMSE: {simple_record['train_rmse']:.2f}; "
      f"validation RMSE: {simple_record['valid_rmse']:.2f}; "
      f"gap: {simple_record['valid_rmse'] - simple_record['train_rmse']:.2f}")
for notice in simple_record['notices']:
    if issubclass(notice.category, ConvergenceWarning):
        print('Note — the simple model did not reach convergence in this run.')

### Manager's question 1

Your data scientist sends you the result above: *training RMSE, validation RMSE, and the gap.*

- How large is the gap between two errors? What can that tell you, and what can it not tell you?
- Is this enough evidence to replace the bank's rules engine?
- What other model comparisons and business outcomes would you request?

Think it through before reading on.

---

**Check your reading.** Similar training and validation errors are useful evidence. The relatively small gap means the model performs about as well on transactions it has never seen as on the ones it trained on — it is generalizing, not memorizing. That is encouraging, but a small gap can also occur when both errors are high. To judge how well the model predicts, we need to consider the error level and compare it with a benchmark. Movement 2 compares alternative neural-network architectures. These comparisons help identify a candidate for further evaluation. They do not, by themselves, establish readiness for deployment.

# Movement 2 — Compare Seven Architectures

Compare one versus three layers, two versus five versus thirty neurons, and sigmoid versus ReLU. Every model uses the same data, preprocessing, and common training settings. You can edit an architecture below and rerun from this cell. Before running the next cell, write down which architecture you expect to have the lowest validation RMSE.

In [ ]:
architectures = [
    ('1 layer, 2 neurons (sigmoid)', (2,), 'logistic'),
    ('1 layer, 5 neurons (sigmoid)', (5,), 'logistic'),
    ('3 layers x 2 neurons (sigmoid)', (2, 2, 2), 'logistic'),
    ('3 layers x 5 neurons (sigmoid)', (5, 5, 5), 'logistic'),
    ('3 layers x 5 neurons (ReLU)', (5, 5, 5), 'relu'),
    ('1 layer, 30 neurons (ReLU)', (30,), 'relu'),
    ('3 layers x 30 neurons (ReLU)', (30, 30, 30), 'relu'),
]
model_records = []
for label, layers, activation in architectures:
    model = MLPRegressor(hidden_layer_sizes=layers, activation=activation, max_iter=1000, **settings)
    with warnings.catch_warnings(record=True) as notices, threadpool_limits(limits=1):
        model.fit(X_train, y_train)
        pt = scaler_y.inverse_transform(model.predict(X_train).reshape(-1, 1)).ravel()
        pv = scaler_y.inverse_transform(model.predict(X_valid).reshape(-1, 1)).ravel()
    model_records.append({'label': label, 'model': model, 'pred_valid': pv,
                          'train_rmse': np.sqrt(mean_squared_error(y[train_idx], pt)),
                          'valid_rmse': np.sqrt(mean_squared_error(y[valid_idx], pv)), 'notices': list(notices)})
results_df = pd.DataFrame([{'Architecture': r['label'], 'Training RMSE': r['train_rmse'],
                           'Validation RMSE': r['valid_rmse'], 'Gap': r['valid_rmse'] - r['train_rmse']}
                          for r in model_records])
display(results_df.round(2))
for r in model_records:
    for notice in r['notices']:
        if issubclass(notice.category, ConvergenceWarning) and 'LIMIT' in str(notice.message).upper():
            display(Markdown(f"*{r['label']} stopped at the training limit; convergence was not reached.*"))
        else:
            print(f"Note — {r['label']}: {notice.category.__name__}: {notice.message}")

### Manager's question 2 — The training-error trap

Your data scientist emails you the table above and writes: *“I've tried seven designs. The 3-layers-by-30-neurons ReLU model has by far the lowest training error. The design with the lowest training error is clearly the best. I recommend deploying it.”*

- Which model wins by training error? Which wins by validation error?
- Does the extra training fit improve performance on held-out transactions?
- What is your reply?

Think it through before reading on.

---

**Check your reading.** The table above is similar to the one you met on the overfitting page. Read both errors and the gap together. A small gap is reassuring only when the errors themselves are reasonably low. When a larger model improves training error but worsens validation error, that is evidence of overfitting.

- Compare the small sigmoid models with the deeper sigmoid models.
- Compare the two deep five-neuron models: what changes with ReLU?
- Do the 30-neuron models' lower training errors carry through to validation?

Three observations sit in one frame:

- **Rows 0–1 (small sigmoid): a good balance of accuracy and simplicity.** Small gaps, relatively low validation errors. The **1 layer, 2 neurons (sigmoid)** model performs nearly as well as the larger **3 layers, 5 neurons (ReLU)** model, which achieves the lowest validation RMSE in the table.
- **Rows 2–3 (deep sigmoid): UNDERFIT — and note *why*.** Both errors are bad, and both deep sigmoid rows post *almost identical* numbers: the model is not learning at all. This is not a model that is too simple; the deep sigmoid design blocks the learning signal itself — which is why row 4, the **same three-layer, five-neuron architecture as row 3** with only the gatekeeper switched to ReLU, is rescued into a good fit. An activation mismatch looks like underfitting but has a different cure.
- **Rows 5–6 (large ReLU): OVERFIT.** The lowest training errors in the table paired with validation errors *worse* than the simple model's. All that capacity went into memorizing the training rows.



The model with the lowest validation RMSE and a relatively small gap is a candidate for further evaluation. If two models have nearly equal validation error, what else would influence your recommendation? Use validation to nominate a candidate, then ask for evidence about actual fraud outcomes and customer consequences.

In [ ]:
best_index = int(np.argmin([r['valid_rmse'] for r in model_records]))
best_record = model_records[best_index]
training_winner = min(model_records, key=lambda r: r['train_rmse'])
print('Lowest training RMSE:', training_winner['label'])
print('Lowest validation RMSE:', best_record['label'])
predicted_flags = best_record['pred_valid'] > 75
print(f'At >75, the selected model would decline and review {predicted_flags.sum()} of {len(valid_idx)} validation transactions.')

The 75 cutoff converts predictions into proposed actions: transactions scoring above 75 are flagged for decline and review. The count above tells us how often the model would act, but not how many flags are correct or how much fraud it misses. What additional evidence would you need to determine whether the model performs better than the existing rules engine?

# Interpreting the Output — Choose Your Path

Complete **Path A or Path B**, or both. Path A examines whether a GenAI explanation can support an individual decision. Path B examines whether the network's predictive performance justifies giving up a simpler model's readable coefficients. Both ask whether you could explain, justify, and stand behind FraudFinder's use of the model. If you choose Path A only, skip the Path B cells and continue to Final Reflection. If you choose Path B only, you may skip the Path A cells.



## Path A — Critique a GenAI Explanation

Imagine a customer calling to ask why their purchase was declined. A high
score tells you what action the system proposes, but it does not itself explain
how the network arrived there. Your task is to assess an explanation you might
be tempted to give that customer.

1. Run the next cell. It selects the **highest-scoring transaction among the
   450 validation transactions** using the Movement 2 candidate. Read its
   features, predicted score, and proposed action. In this case, a score
   **above 75** triggers decline and review; it does not establish that fraud occurred.
2. Copy the displayed features, model name, score, and action into a
   general-purpose GenAI tool with this prompt:

   > Our bank's neural network assigned this transaction the score and action shown below. Why might this transaction look suspicious? Explain in language a customer could understand.

3. Keep the tool's response and write a brief critique **before running the
   Afterword near the end of the notebook**. Address these questions:
   - What did the explanation describe well? Which claims are plausible
     hypotheses, and which does the supplied information actually support?
   - Did the tool examine the fitted network, or reason from the features you
     pasted? What evidence would you need to check its claims about the score?
   - Could the score be wrong even if the explanation sounds persuasive?
     Would you stand behind this explanation with a customer or reviewer?

The GenAI tool receives the transaction and prediction, not the fitted network or its
learned weights. Evaluate what its response can establish about this particular
model. If it acknowledges that limitation, recognize that as a strength. The
expected result is a reasoned critique, not proof that the transaction is fraud
or that the tool's explanation must be wrong.

In [ ]:
flag_position = int(np.argmax(best_record['pred_valid']))
selected_row_id = int(valid_idx[flag_position])
predicted_score = float(best_record['pred_valid'][flag_position])
print(f"Model: {best_record['label']}; CSV data row: {selected_row_id + 1}")
print(f'Raw predicted score: {predicted_score:.2f}; bounded display: {np.clip(predicted_score, 0, 100):.2f}')
print('Action:', 'Decline and review' if predicted_score > 75 else 'Allow')
display(df.iloc[selected_row_id].drop('risk_score').to_frame('Transaction'))

**Check your reading.** Neural network predictions can overshoot 0–100. The displayed operational score is bounded to that range; RMSE throughout uses the raw predictions. Decisions use the unrounded score. The reference score is revealed in the Afterword, after you have completed your critique.

## Path B — Compare with Linear Regression

FraudFinder's board wants to know whether a harder-to-explain model provides
enough predictive improvement to justify using it. Compare the Movement 2
candidate with **linear regression**, the kind of model you met in Week 5.
We use linear regression because the target is a continuous suspicion score.

Run the next two cells. Both models use the **same 1,050 training rows, the same
450 validation rows, and the same encoded and scaled inputs**.

First, read the two validation RMSE values. The printed difference is
**linear-regression validation RMSE − neural-network validation RMSE**, in
score points. A positive value favors the network; a negative value favors
linear regression; a value near zero indicates little difference on this sample.
Then inspect the coefficient examples and write a short response addressing three questions:

1. How much did the network improve validation RMSE over linear regression,
   if at all? Cite both values and the difference in score points. Use your
   output rather than assuming the network must win.
2. Interpret one or two displayed coefficients in plain language, including
   the unit or comparison category. What can you explain directly about the
   linear model that the network's score alone does not tell you?
3. Is the observed predictive improvement worth the added explanation burden
   for FraudFinder? Name a customer or board question that this comparison
   still cannot answer, and what evidence would help answer it.

In [ ]:
with threadpool_limits(limits=1):
    linreg = LinearRegression().fit(X_train, y_train)
    lr_pred_train = scaler_y.inverse_transform(linreg.predict(X_train).reshape(-1, 1)).ravel()
    lr_pred_valid = scaler_y.inverse_transform(linreg.predict(X_valid).reshape(-1, 1)).ravel()
lr_train = np.sqrt(mean_squared_error(y[train_idx], lr_pred_train))
lr_valid = np.sqrt(mean_squared_error(y[valid_idx], lr_pred_valid))
display(pd.DataFrame([
    {'Model': 'Linear regression', 'Training RMSE': lr_train, 'Validation RMSE': lr_valid},
    {'Model': best_record['label'], 'Training RMSE': best_record['train_rmse'], 'Validation RMSE': best_record['valid_rmse']}
]).round(2))
print(f"Linear-regression error minus network error: {lr_valid - best_record['valid_rmse']:.2f} points; positive favors the network.")

In [ ]:
# Coefficients expressed in score points per original predictor unit.
coefficients_raw = linreg.coef_ * scaler_X.scale_ / scaler_y.scale_[0]
coefficients = pd.DataFrame({'Risk points per unit': coefficients_raw}, index=X.columns)
display(coefficients.round(4))

**Check your reading.** Each linear-regression coefficient gives the change in predicted score for a
one-unit increase in that input, with other inputs held fixed. A positive
coefficient raises the predicted score; a negative one lowers it. For example,
the `txn_count_24h` coefficient is the change in score points for one additional
prior authorization attempt in the last 24 hours.

For a 0/1 indicator, the coefficient describes changing from 0 to 1.
Categorical coefficients compare with **Electronics**, **Chip**, and **High
income**. The Online indicator is the Online-versus-Chip contrast. Different units mean coefficient magnitudes are not an importance ranking.

A readable coefficient describes the fitted model's calculation; it does not
establish causation or prove that a fraud decision is correct. Your conclusion
should weigh the observed predictive difference against the need to explain
individual decisions.



# Final Reflection — Your Recommendation to the Board

Return to the CFO's proposal to replace the rules engine. Write **one paragraph**
that draws on your Movement 2 comparison table and **the interpretation path you
completed**. You are recommending the next justified step for FraudFinder.
Include the following:

- **Your candidate and numerical evidence.** Name the architecture you would
  advance for further evaluation. Cite its training and validation RMSE, and
  explain why the lowest training error alone does not decide the choice.
  If candidates are nearly tied on validation error, explain your preference.
- **What your chosen path adds.** For Path A, use a finding from your critique
  to explain what you could or could not defend about a transaction. For Path B,
  use the measured difference from linear regression and what its coefficients
  make easier to explain. You need evidence from only the path you completed.
- **A limitation and a condition for proceeding.** State one practical action
  or piece of evidence you would require before the model decides live
  transactions. Explain how that condition bears on your recommendation.
- **The comparison the board still needs.** The rules engine catches about
  **62% of fraud** and wrongly declines about **8% of legitimate purchases**.
  What would FraudFinder have to measure to compare the proposed network
  fairly with those two numbers?

Keep these as parts of one recommendation, rather than four separate answers.
The notebook measures error in predicting synthetic scores and contains no
confirmed fraud outcomes. It therefore cannot establish the network's fraud
catch rate or wrongful-decline rate. Also, the validation data used to select
the model are not a final independent test. Distinguish what your results
support now from what would require further evaluation.

In [ ]:
# Afterword: run after the Path A critique; also works if you chose only Path B.
flag_position = int(np.argmax(best_record['pred_valid']))
selected_row_id = int(valid_idx[flag_position])
predicted_score = float(best_record['pred_valid'][flag_position])
actual_score = float(y[selected_row_id])
model_action = 'decline and review' if predicted_score > 75 else 'allow'
reference_action = 'decline and review' if actual_score > 75 else 'allow'
display(Markdown(f"""
## Afterword — Check the Same Transaction

For **data row {selected_row_id + 1}**, the raw prediction is **{predicted_score:.2f}**
and the reference score is **{actual_score:.1f}**: an error of **{predicted_score - actual_score:+.2f} points**.
The model's action is **{model_action}**; the reference-score action is **{reference_action}**.

If you completed Path A, did the assistant distinguish a plausible explanation
from evidence about this fitted model?

For either path, a high suspicion score is not a probability of fraud, and
agreement with a synthetic reference does not establish whether fraud occurred.
"""))

# Before You Leave

Save your Colab copy with its outputs and written responses. **The notebook
itself is not submitted.** Keep the following ready for the live session:

- Your **Movement 2 architecture comparison table**, with training and
  validation RMSE side by side, and the candidate you would advance.
- Your **one-paragraph board recommendation** and the main finding from your
  chosen interpretation path, including a question that path left unanswered.
- Any question or point where you got stuck. Note the section and the output
  so you can show exactly what you could produce but could not yet explain.

Return to the **Week 7 Wrap-Up** with the first response to the CFO's dilemma
that you saved at the start of the week. What can you now assess using evidence?
What has changed in your judgment, and what is still missing before the neural
network's claim can be compared with the rules engine's measured results?
Use that comparison when completing the wrap-up reflection.

## Optional — What If We Had Less Data?

Movement 2 changed the network's architecture while keeping the training data
fixed. Now ask a different question: **what happens to each architecture when
it has fewer examples to learn from?** A network trained on fewer rows still
has the same layers, neurons, and activation function. Fewer training examples
and fewer neurons are two different changes.

This extension uses a **400-row subset**: 280 training rows and 120 validation
rows drawn from the original split. The cell retrains all seven architectures
on the 280 training rows. It then compares them with the models already trained
on 1,050 rows in Movement 2. Each model's scaling is learned from its own
training rows only.

**How to read the table:** read across a row to compare the same architecture
trained on 1,050 versus 280 examples. Read down a validation column to compare
different architectures given the same amount of training data. Lower
validation RMSE means more accurate predictions on the held-out transactions.

Both validation columns use the **same 120 transactions**, making the two
training-data sizes comparable. Movement 2 evaluated 450 validation transactions,
so its validation RMSE values can differ from the ones here, even for the
unchanged models trained on 1,050 rows.

Each **RMSE gap = validation RMSE on these 120 transactions − training RMSE on
that model's own training rows**. A large positive gap means training error
was much lower than validation error. Read the gap together with the error
level; a small gap alone does not establish a good model.

In [ ]:
small_train_idx = np.random.RandomState(41).choice(train_idx, size=280, replace=False)
common_valid_idx = np.random.RandomState(42).choice(valid_idx, size=120, replace=False)
small_scaler_X, small_scaler_y = MinMaxScaler(), MinMaxScaler()
small_X_train = small_scaler_X.fit_transform(X.iloc[small_train_idx])
small_X_valid = small_scaler_X.transform(X.iloc[common_valid_idx])
small_y_train = small_scaler_y.fit_transform(y[small_train_idx].reshape(-1, 1)).ravel()
common_positions = pd.Index(valid_idx).get_indexer(common_valid_idx)
small_rows, small_records = [], []
for (label, layers, activation), full_record in zip(architectures, model_records):
    small_model = MLPRegressor(hidden_layer_sizes=layers, activation=activation, max_iter=1000, **settings)
    with warnings.catch_warnings(record=True) as notices, threadpool_limits(limits=1):
        small_model.fit(small_X_train, small_y_train)
        small_pt = small_scaler_y.inverse_transform(small_model.predict(small_X_train).reshape(-1, 1)).ravel()
        small_pv = small_scaler_y.inverse_transform(small_model.predict(small_X_valid).reshape(-1, 1)).ravel()
    small_train_error = np.sqrt(mean_squared_error(y[small_train_idx], small_pt))
    small_valid_error = np.sqrt(mean_squared_error(y[common_valid_idx], small_pv))
    full_valid_error = np.sqrt(mean_squared_error(y[common_valid_idx], full_record['pred_valid'][common_positions]))
    small_rows.append({'Architecture': label, 'Validation RMSE (1,050 training rows)': full_valid_error,
                       'Validation RMSE (280 training rows)': small_valid_error, 'RMSE gap (1,050 training rows)': full_valid_error - full_record['train_rmse'],
                       'RMSE gap (280 training rows)': small_valid_error - small_train_error})
    small_records.append({'model': small_model, 'train_rmse': small_train_error, 'valid_rmse': small_valid_error,
                          'pred_valid': small_pv, 'notices': list(notices)})
display(pd.DataFrame(small_rows).round(2))
for (label, _, _), record in zip(architectures, small_records):
    for notice in record['notices']:
        if issubclass(notice.category, ConvergenceWarning) and 'LIMIT' in str(notice.message).upper():
            display(Markdown(f'*{label} stopped at the training limit; convergence was not reached.*'))
        else:
            print(f'Note — {label}: {notice.category.__name__}: {notice.message}')

**Check your reading.** Work through the table in this order and record a few
sentences about what you find:

1. Start with **1 layer, 5 neurons (sigmoid)**. Read across its row: did
   validation RMSE rise or fall when training rows fell from 1,050 to 280?
   How did its RMSE gap change? You are comparing the same design with
   different amounts of training data.
2. Repeat that comparison for **3 layers × 30 neurons (ReLU)**. Does the
   larger network handle the reduction in data as well as the small sigmoid
   network? A widening gap together with poorer validation performance
   strengthens the concern about overfitting.
3. Scan the remaining rows. Find an architecture that behaves differently
   from those two, and describe the difference. Check the error level as well
   as the gap: high validation RMSE with a small gap is still poor prediction.

Fewer examples can make it harder for a flexible network to generalize, but
one subsample need not make every architecture's error or gap worse. Learning
and optimization can also change with the sample. Use the results to judge
whether an architecture's extra flexibility is supported by the available data;
do not treat a single improvement as evidence that less data is generally better.